# Practical Session 11: Dimensionality Reduction with PCA and LDA

This notebook is a fully documented model solution for the current practical script. It
generates synthetic multi-class data, performs PCA manually and with a toolbox, studies
reconstruction error, applies LDA, and compares both approaches quantitatively.


In [ ]:
# Import NumPy for linear algebra and synthetic data generation.
import numpy as np
# Import pandas for compact comparison tables.
import pandas as pd
# Import Matplotlib for all visualizations.
import matplotlib.pyplot as plt

# Import scikit-learn implementations for PCA, LDA, and classification.
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Print arrays compactly in notebook output.
np.set_printoptions(precision=4, suppress=True)

# Fix the random seed so the synthetic data and all experiments are reproducible.
SEED = 11
rng = np.random.default_rng(SEED)


## Task 1: Generate synthetic data

We construct a three-class data set in dimension five with correlated features, one small
variance direction, and moderate class overlap.


In [ ]:
def generate_synthetic_data(noise_scale=1.0, n_per_class=90, seed=SEED, d=5):
    # Use a local generator so the function stays reproducible for any given seed.
    local_rng = np.random.default_rng(seed)

    # Define distinct class means in the first few informative coordinates.
    base_means = np.array(
        [
            [2.5, 0.0, 0.8, -0.5, 0.1],
            [-1.0, 2.0, -0.6, 0.4, -0.1],
            [0.5, -2.2, 1.3, 0.0, 0.2],
        ]
    )

    # Build a covariance matrix with correlated features and one low-variance direction.
    covariance = np.array(
        [
            [1.2, 0.75, 0.20, 0.10, 0.00],
            [0.75, 1.0, 0.15, 0.05, 0.00],
            [0.20, 0.15, 0.9, 0.25, 0.00],
            [0.10, 0.05, 0.25, 0.7, 0.00],
            [0.00, 0.00, 0.00, 0.00, 0.08],
        ]
    )

    # Allow an optional higher-dimensional extension by padding with low-noise features.
    if d > 5:
        extra = d - 5
        padded_means = np.hstack([base_means, np.zeros((3, extra))])
        padded_cov = np.eye(d) * 0.03
        padded_cov[:5, :5] = covariance
        covariance = padded_cov
        base_means = padded_means
        # Inject a few latent directions with stronger structure.
        for cls in range(3):
            base_means[cls, 5 : min(d, 8)] = np.array([1.5, -1.0, 0.8])[: max(0, min(d, 8) - 5)] * (cls - 1)

    # Scale the covariance to control the overlap between the classes.
    covariance = covariance * noise_scale

    # Sample each class from a multivariate normal distribution.
    X_parts = []
    y_parts = []
    for class_id, mean in enumerate(base_means):
        samples = local_rng.multivariate_normal(mean, covariance, size=n_per_class)
        X_parts.append(samples)
        y_parts.append(np.full(n_per_class, class_id))

    # TODO: Stack all class-specific sample matrices into one data matrix.
X = ...
    y = np.concatenate(y_parts)
    return X, y, covariance


# Generate the main synthetic data set from the practical specification.
X, y, covariance = generate_synthetic_data()
print("Data matrix shape:", X.shape)
print("Label vector shape:", y.shape)

# Visualize two original features to show the class overlap.
plt.figure(figsize=(6, 4))
for class_id in np.unique(y):
    plt.scatter(
        X[y == class_id, 0],
        X[y == class_id, 1],
        label=f"class {class_id}",
        alpha=0.7,
    )
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Synthetic data in the original feature space")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Task 2: PCA from the covariance matrix

We center the data, build the empirical covariance matrix, compute the eigendecomposition,
and project the data onto the first two principal components.


In [ ]:
# Compute the empirical mean and center all samples.
x_mean = X.mean(axis=0)
# TODO: Center the data matrix by subtracting the empirical mean.
X_centered = ...

# Build the empirical covariance matrix S = (1/N) X_c^T X_c.
S = (X_centered.T @ X_centered) / len(X_centered)

# Compute eigenvalues and eigenvectors of the symmetric covariance matrix.
eigenvalues, eigenvectors = np.linalg.eigh(S)

# Sort the components by descending explained variance.
# TODO: Sort the eigenvalues in descending order.
order = ...
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

# Project the centered data to the first two principal components.
U2 = eigenvectors[:, :2]
Z_pca_manual = X_centered @ U2

# Compute explained variance ratios of all components.
explained_variance_ratio = eigenvalues / eigenvalues.sum()
print("Explained variance ratios:", explained_variance_ratio)

# Visualize the manually computed two-dimensional PCA projection.
plt.figure(figsize=(6, 4))
for class_id in np.unique(y):
    plt.scatter(
        Z_pca_manual[y == class_id, 0],
        Z_pca_manual[y == class_id, 1],
        label=f"class {class_id}",
        alpha=0.7,
    )
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.title("Manual PCA projection")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Task 3: PCA with a toolbox

We repeat the PCA experiment with scikit-learn and compare directions, explained variance,
and projected coordinates with the manual version.


In [ ]:
# Create a scikit-learn PCA object for the same target dimension.
pca_toolbox = PCA(n_components=2, random_state=SEED)
Z_pca_toolbox = pca_toolbox.fit_transform(X)

# Compare the explained variance ratios and the projection coordinates.
pca_compare_df = pd.DataFrame(
    {
        "manual_explained_variance_ratio": explained_variance_ratio[:2],
        "toolbox_explained_variance_ratio": pca_toolbox.explained_variance_ratio_,
    },
    index=["component_1", "component_2"],
)
display(pca_compare_df)

# Compare principal directions up to sign differences.
direction_alignment = np.abs(np.sum(U2 * pca_toolbox.components_.T, axis=0))
print("Absolute alignment of principal directions:", direction_alignment)
print("Maximum absolute coordinate difference:", np.max(np.abs(np.abs(Z_pca_manual) - np.abs(Z_pca_toolbox))))


## Task 4: Reconstruction error in PCA

We reconstruct the original data from the first `k` principal components and measure how
the mean squared reconstruction error decreases as `k` increases.


In [ ]:
reconstruction_rows = []
for k in [1, 2, 3, 4]:
    # Keep the first k principal directions.
    Uk = eigenvectors[:, :k]
    # Compute the low-dimensional coordinates in the retained subspace.
    Zk = X_centered @ Uk
    # Reconstruct the original data approximation and add the mean back.
    # TODO: Reconstruct the data from the first k principal components.
X_hat = ...
    mse = np.mean((X - X_hat) ** 2)
    reconstruction_rows.append({"k": k, "reconstruction_mse": mse})

reconstruction_df = pd.DataFrame(reconstruction_rows)
display(reconstruction_df)

plt.figure(figsize=(6, 4))
plt.plot(reconstruction_df["k"], reconstruction_df["reconstruction_mse"], marker="o")
plt.xlabel("Number of retained principal components")
plt.ylabel("Mean squared reconstruction error")
plt.title("PCA reconstruction error")
plt.grid(True, alpha=0.3)
plt.show()


## Task 5: LDA projection

LDA uses the class labels and can therefore optimize class separation directly. With three
classes, at most `K - 1 = 2` informative discriminant directions exist.


In [ ]:
# Fit an LDA model and reduce the data to two discriminant directions.
lda = LinearDiscriminantAnalysis(n_components=2)
# TODO: Fit LDA and transform the data to the discriminant space.
Z_lda = ...

# Visualize the LDA projection for comparison with PCA.
plt.figure(figsize=(6, 4))
for class_id in np.unique(y):
    plt.scatter(
        Z_lda[y == class_id, 0],
        Z_lda[y == class_id, 1],
        label=f"class {class_id}",
        alpha=0.7,
    )
plt.xlabel("LD 1")
plt.ylabel("LD 2")
plt.title("LDA projection")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Task 6: Quantitative comparison of PCA and LDA

We compare both methods with the cross-validated classification accuracy of a simple
logistic-regression classifier in the projected space.


In [ ]:
# Use a fixed cross-validation split for both projected data sets.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
classifier = LogisticRegression(max_iter=2000)

# Evaluate classification performance after PCA projection.
# TODO: Evaluate the classifier after PCA projection.
pca_scores = ...
# Evaluate classification performance after LDA projection.
lda_scores = cross_val_score(classifier, Z_lda, y, cv=cv)

comparison_df = pd.DataFrame(
    [
        {
            "method": "PCA",
            "mean_cv_accuracy": pca_scores.mean(),
            "std_cv_accuracy": pca_scores.std(),
        },
        {
            "method": "LDA",
            "mean_cv_accuracy": lda_scores.mean(),
            "std_cv_accuracy": lda_scores.std(),
        },
    ]
)
display(comparison_df)


## Task 7: Sensitivity to noise

We increase the isotropic noise level in the synthetic data generation process and compare
how strongly PCA and LDA are affected.


In [ ]:
# Generate a noisier version of the same synthetic problem.
X_noisy, y_noisy, _ = generate_synthetic_data(noise_scale=2.0, seed=SEED + 1)

# Repeat PCA and LDA on the noisier data.
pca_noisy = PCA(n_components=2, random_state=SEED)
Z_pca_noisy = pca_noisy.fit_transform(X_noisy)

lda_noisy = LinearDiscriminantAnalysis(n_components=2)
Z_lda_noisy = lda_noisy.fit_transform(X_noisy, y_noisy)

# Compare explained variance and projected classification accuracy.
pca_noisy_scores = cross_val_score(classifier, Z_pca_noisy, y_noisy, cv=cv)
lda_noisy_scores = cross_val_score(classifier, Z_lda_noisy, y_noisy, cv=cv)

noise_df = pd.DataFrame(
    [
        {
            "setting": "original / PCA",
            "leading_variance_sum": pca_toolbox.explained_variance_ratio_.sum(),
            "mean_cv_accuracy": pca_scores.mean(),
        },
        {
            "setting": "noisy / PCA",
            "leading_variance_sum": pca_noisy.explained_variance_ratio_.sum(),
            "mean_cv_accuracy": pca_noisy_scores.mean(),
        },
        {
            "setting": "original / LDA",
            "leading_variance_sum": np.nan,
            "mean_cv_accuracy": lda_scores.mean(),
        },
        {
            "setting": "noisy / LDA",
            "leading_variance_sum": np.nan,
            "mean_cv_accuracy": lda_noisy_scores.mean(),
        },
    ]
)
display(noise_df)


## Task 8: Optional extension with a higher-dimensional data set

We generate a second synthetic data set in dimension twenty and inspect the PCA spectrum to
see whether the intrinsic dimensionality becomes visible.


In [ ]:
# Generate a second data set with 20 observed dimensions and only a few strong latent directions.
X_hd, y_hd, _ = generate_synthetic_data(noise_scale=1.0, n_per_class=90, seed=SEED + 2, d=20)

# Fit PCA to the high-dimensional data.
pca_hd = PCA(random_state=SEED)
pca_hd.fit(X_hd)
X_hd_proj = pca_hd.transform(X_hd)[:, :3]

# Plot the eigenvalue decay to inspect intrinsic dimensionality.
plt.figure(figsize=(6, 4))
plt.plot(np.arange(1, len(pca_hd.explained_variance_) + 1), pca_hd.explained_variance_, marker="o")
plt.xlabel("Principal component index")
plt.ylabel("Eigenvalue")
plt.title("Eigenvalue decay in the higher-dimensional extension")
plt.grid(True, alpha=0.3)
plt.show()

print("Projected high-dimensional data shape:", X_hd_proj.shape)


## Task 9: Short technical reflection

PCA maximizes projected variance and therefore focuses on compact low-dimensional
representation without using labels. LDA maximizes class separation relative to within-class
scatter and is therefore supervised. A projection with large variance can still be poor for
discrimination when the large-variance direction does not align with the separation of class
means. PCA is often preferable for compression and unsupervised exploration, while LDA is
preferable when labeled classes and discrimination quality are the primary concern.
